# 35 — Caption + OCR cho BATCH 2 (Gemini 3.8 qua cổng Trammay)

Giống hệt nb34 về đường ống, khác ở **một điều duy nhất**: batch 2 **không có OCR nào** — không `qwen_ocr.json`, không `vietocr`. Nên prompt đổi sang `v5_bao_ocr` (hai khối `[FINAL_CAPTION]` + `[OCR]`) và bản gộp xuất ra **hai** file.

**Vì sao phải có OCR riêng, không để nhúng trong caption:** hai arm `ocr_qwen25` và `ocr_vietocr` đang cho điểm. Không có trường OCR thì với MỌI keyframe batch 2 chúng trả về rỗng — không nổ, không cảnh báo, chỉ là khung batch 2 nhận ít phiếu hơn khung batch 1 trong hợp nhất. Đó là **thiên lệch xếp hạng giữa hai nửa kho**.

**Chữ trên màn hình xuất HAI LẦN** — vừa trong caption vừa trong khối `[OCR]`. Trùng lặp là cố ý: rút hẳn OCR khỏi caption thì caption batch 2 khác cấu trúc caption batch 1 (bản sinh 22-09 có OCR nhúng), và hai nửa lại lệch ở chỗ khác.

**Part lưu chuỗi THÔ, tách ở AGGREGATE.** Tách sai thì re-split từ part tốn 0 đồng; sinh lại cả kho tốn ~320 triệu token. Bản thô cũng được đẩy lên volume (`gemini_raw.json`) vì lý do đó.

## ⚠ TRƯỚC KHI CHẠY — ba con số phải điền

`EXPECTED_VIDEOS`, `EXPECTED_FRAMES` và `BTC_BUNDLE` đang là giá trị giữ chỗ, và ô cấu hình sẽ **ném lỗi** nếu chưa điền. Lấy chúng bằng:

```bash
.venv/Scripts/python.exe scratchpad/kiem_bat_bien_batch2.py --bundle <ten_bundle>
```
Script đó cũng kiểm luôn `frame_idx == floor(pts_time × fps)` và đếm cặp trùng `frame_idx`. **Đừng giả định batch 2 giống batch 1** — batch 1 có đúng 4 giá trị fps, 614 cặp trùng và một video chạy 26,44 fps, không cái nào suy ra được.

**Quota:** batch 2 cỡ batch 1 thì cần ~22.000 request + ~320 triệu token. Trần là 10.000 request và 100 triệu token **mỗi user mỗi ngày** ⇒ tối thiểu **4 key**.
## Trước khi Run All

1. **Accelerator: None (CPU)** — notebook không nạp model nào.

2. **Internet: ON** — cần cho `git clone`, `pip install modal` và gọi API.

3. **Add-ons → Secrets**, đủ **bốn** tên (mỗi tài khoản Kaggle tự thêm):

   | secret | dùng làm gì |
   |---|---|
   | `GITHUB_TOKEN` | clone repo — repo này **private**, thiếu token là notebook TREO ở bước clone chứ không báo lỗi |
   | `TRAMMAY_API_KEY` | gọi cổng caption |
   | `MODAL_TOKEN_ID` | đẩy kết quả lên Volume |
   | `MODAL_TOKEN_SECRET` | ditto |

4. **KHÔNG cần gắn dataset nào.** Ảnh và `map-keyframes` kéo thẳng từ Modal volume `keyframes-btc`, giống nb25/26/33. Gắn dataset Kaggle là mở một nguồn sự thật thứ hai: sáu worker gắn sáu dataset khác nhau sẽ chia việc khác nhau, và có video **không ai làm** mà `SHARD` nào cũng báo COMPLETE.

## ⚠ Ba chỗ đã trả giá, đừng gỡ

**1. Khoá là `frame_idx`, KHÔNG phải `n`.** Notebook thử nghiệm trước đặt khoá `frame_{n:03d}`. `n` là số thứ tự keyframe *và* tên file ảnh; `frame_idx` là khung thật trong video, và `map-keyframes/{video}.csv` là chỗ duy nhất nối hai số đó. Nhầm chúng là cách dễ nhất để caption gắn lên khung khác mà mọi thứ trông vẫn đúng.

**2. API key đọc từ Kaggle Secrets, KHÔNG viết cứng.** Notebook tham chiếu từng có key nguyên văn trong cell 2.

**3. `BATCH_MAX_BYTES` là hàng rào thật, không phải `BATCH_SIZE`.** Cổng chặn theo kích thước thân request; trần là **8 MB** (đo 22-09: 5 MB qua, 8 MB trả 413). Nhưng ảnh nặng rất khác nhau giữa hai batch và giữa các nhóm nội dung — đo 23-09 trên 208.491 keyframe batch 2:

   | nguồn | KB/ảnh | base64 | thân request khi k=8 |
   |---|---:|---:|---:|
   | batch 1 | 178 | 237 KB | 1,85 MiB |
   | batch 2 · M tin tức | 388 | 517 KB | 4,04 MiB |
   | batch 2 · S đua xe | 393 | 524 KB | 4,10 MiB |
   | batch 2 · **N giao thông** | **582** | **776 KB** | **6,06 MiB** |

   Ảnh giao thông **gấp 3,3 lần batch 1** — 1080p đầy chi tiết tần số cao (mặt đường, nhiều vật nhỏ, nhiễu đêm) nên JPEG nén kém. `_chia_batch` cắt theo **byte trước, số lượng sau**, nên đặt `BATCH_SIZE` lớn hơn mức byte cho phép sẽ **không** ra batch lớn hơn — chỉ làm con số trong config nói dối.
**4. `BACKOFF_BASE = 30` chứ không phải 2 — đây là thứ đưa tỉ lệ hỏng về 0.** Trần đồng thời của cổng đếm request **đang bay**, mà một batch 8 ảnh chạy 23–47s. Lùi 2s/4s/8s là thử lại vào đúng lúc các suất vẫn chưa trống nên cả bốn lần đều trượt. Đo trên cùng 140 khung: nền 2s + 4 luồng → **8/18 batch chết**; nền 2s + 2 luồng → 1/18; nền 30s + 2 luồng → **0/18, 140/140 caption**.

**5. `MAX_WORKERS = 2` là số luồng MỖI worker, và sáu worker dùng CHUNG một trần.** Con số 0/18 ở trên đo từ **một máy, 2 request đồng thời**. Sáu worker × 2 luồng = **12 đồng thời** vào cùng một trần mà ta **chưa biết con số**. Đừng thả cả sáu cùng lúc: cho một người chạy trước, nhìn dòng `hỏng` của 2–3 video đầu, rồi thêm từng người một. `hỏng` vọt lên là đã chạm trần — lùi lại số worker trước đó.

**6. Phụ lục ép độ dài KHÔNG được gỡ.** Batch làm model tự chia phần và caption tụt 195 → 148 từ; phụ lục kéo về 197. Số đo nằm ở ô nạp prompt.


In [ ]:
from pathlib import Path

# ── ai chạy, chạy phạm vi nào ──────────────────────────────────────────────
RUNNER = "w1"                     # w1..w6
RUNNERS = ("w1", "w2", "w3", "w4", "w5", "w6")
RUN_SCOPE = "PREFLIGHT"           # PREFLIGHT | SHARD | AGGREGATE
CONFIRM_SHARD = ""                # SHARD: đặt đúng bằng RUNNER
CONFIRM_AGGREGATE = False         # AGGREGATE: True sau khi cả 6 shard COMPLETE

# ── cổng API ───────────────────────────────────────────────────────────────
# Kaggle Secrets: Add-ons -> Secrets -> TRAMMAY_API_KEY
TRAMMAY_SECRET = "TRAMMAY_API_KEY"
MODEL_NAME = "ag/gemini-3.8-flash"
PROMPT_VARIANT = "v5_bao_ocr"   # HAI khối: [FINAL_CAPTION] + [OCR]
# Cổng BỎ QUA `max_tokens` (đo 22-09: gửi 40 vẫn trả 263 token, `finish_reason
# ='stop'`), nên con số này chỉ để qua cửa kiểm tra. Trần của endpoint batch là
# 8.192; chat thường vẫn 4.096.
MAX_TOKENS = 8192
# ── đường gọi: "chat" (1 ảnh/1 request) hay "batch" (k ảnh/1 request) ───────
#
# Chọn theo thứ đang CHẶN, không theo thứ nghe nhanh hơn:
#
#   "batch"  khi trần REQUEST chặn (10.000/ngày/user). Giảm k lần số request,
#            đổi lại chậm hơn 1,47 lần và caption cần phụ lục `len180` để không
#            tụt từ 195 xuống 148 từ.
#   "chat"   khi trần request đã nới. Nhanh hơn (2.040 so 1.388 khung/giờ/
#            worker) và tái lập ĐÚNG cấu hình đã sinh 499 video đêm 21-09.
#
# ⚠ `BATCH_SIZE = 1` KHÔNG tương đương `"chat"`: nó vẫn đi qua
# `/chat/completions/batch`, tức vẫn dính system prompt ép JSON mà cổng chèn —
# chính thứ làm model tự chia phần. Muốn một-ảnh-một-request thật thì phải là
# `"chat"`.
DUONG_GOI = "chat"                # "chat" | "batch"
MAX_TOKENS_CHAT = 4096            # trần endpoint thường; batch được 8192
MOC_GHI = 80 if DUONG_GOI == "chat" else 10   # ghi checkpoint mỗi ~80 ảnh
BATCH_SIZE = 8                    # chỉ dùng khi DUONG_GOI == "batch"
BATCH_MAX_BYTES = 7_000_000       # hàng rào THẬT: cắt theo byte — xem bảng dưới
# Số luồng và nền lùi phải khớp ĐƯỜNG GỌI — cả hai đều theo độ dài một request:
#   "chat"  : ~8s/lượt  -> 28 luồng, lùi 2s   (đo đêm 21-09: 6–9 giờ sạch)
#   "batch" : 23–47s    ->  2 luồng, lùi 30s  (đo 22-09: 0/18 hỏng)
# Đặt lẫn là hỏng theo hai kiểu: 28 luồng batch thì 524 hàng loạt, còn 2 luồng
# chat thì chậm gấp 14 lần mà chẳng an toàn hơn.
MAX_WORKERS = 28 if DUONG_GOI == "chat" else 2
BACKOFF_BASE = 2.0 if DUONG_GOI == "chat" else 30.0
REQUEST_TIMEOUT = 300
MAX_RETRIES = 4

# ── đường dẫn ──────────────────────────────────────────────────────────────
REPO_URL = "https://github.com/AIVIETNAM-AIO-Triet-Descartes/multimodal-fusion-engine.git"
REPO_REF = "feature/triet/structure"
REPO_DIR = Path("/kaggle/working/multimodal-fusion-engine")
WORK_DIR = Path("/kaggle/working/caption-gemini")

# Nguồn ảnh + map-keyframes: volume BTC, KHÔNG phải `/kaggle/input`.
BTC_VOLUME = "keyframes-btc"
BTC_BUNDLE = "aic_btc_b2_v1"
CACHE_ANH = WORK_DIR / "keyframes"

# Gốc RIÊNG cho batch 2. Khác batch 1 ở prompt (hai khối) và ở nội dung,
# nên trộn chung một gốc là mất khả năng phân biệt hai nửa về sau.
OUTPUT_ROOT = "/research/caption-ocr-b2-v1"
RESEARCH_VOLUME = "keyframe-augmentation-research"

EXPECTED_VIDEOS = 0
EXPECTED_FRAMES = 0          # ĐẾM bằng kiem_bat_bien_batch2.py, đừng chép của batch 1

if EXPECTED_VIDEOS <= 0 or EXPECTED_FRAMES <= 0:
    raise RuntimeError(
        "EXPECTED_VIDEOS/EXPECTED_FRAMES còn là 0 — chạy "
        "scratchpad/kiem_bat_bien_batch2.py rồi chép hai con số nó in ra vào đây. "
        "Chia việc tính trên chúng; sai là có video không ai làm."
    )
if RUNNER not in RUNNERS:
    raise ValueError(f"RUNNER phải thuộc {RUNNERS}")
if RUN_SCOPE not in {"PREFLIGHT", "SHARD", "AGGREGATE"}:
    raise ValueError("RUN_SCOPE phải là PREFLIGHT, SHARD hoặc AGGREGATE")
if RUN_SCOPE == "SHARD" and CONFIRM_SHARD != RUNNER:
    raise RuntimeError("SHARD bị khoá; đặt CONFIRM_SHARD đúng bằng RUNNER")
if RUN_SCOPE == "AGGREGATE" and not CONFIRM_AGGREGATE:
    raise RuntimeError("AGGREGATE bị khoá; đặt CONFIRM_AGGREGATE = True")
if DUONG_GOI not in {"chat", "batch"}:
    raise ValueError('DUONG_GOI phải là "chat" hoặc "batch"')
if not 1 <= BATCH_SIZE <= 8:
    raise ValueError("BATCH_SIZE phải từ 1 đến 8")
if BATCH_MAX_BYTES < 700_000 or BATCH_MAX_BYTES > 7_500_000:
    # Trần thân request của cổng là 8 MB (Đạt nâng 22-09; đo: 5 MB qua, 8 MB
    # trả 413). 7,5 MB là mức cao nhất còn chừa biên cho overhead JSON.
    #
    # ⚠ KÍCH THƯỚC ẢNH KHÁC NHAU RẤT NHIỀU giữa hai batch và giữa các nhóm nội
    # dung — đo 23-09 trên 208.491 keyframe batch 2 (zip / số khung, theo worker):
    #
    #     batch 1              178 KB/ảnh  -> base64 237 KB -> k=8 = 1,85 MiB
    #     batch 2 M tin tức    388 KB      -> 517 KB        -> k=8 = 4,04 MiB
    #     batch 2 S đua xe     393 KB      -> 524 KB        -> k=8 = 4,10 MiB
    #     batch 2 N giao thông 582 KB      -> 776 KB        -> k=8 = 6,06 MiB
    #
    # Ảnh giao thông gấp 3,3 lần batch 1: 1080p đầy chi tiết tần số cao (mặt
    # đường, nhiều vật nhỏ, nhiễu đêm) nên JPEG nén kém. Để 4,5 MB như trước là
    # batch nhóm N tự cắt xuống ~5 ảnh, và `BATCH_SIZE = 8` thành con số nói dối.
    #
    # 7,0 MB chừa ~1 MB biên dưới trần 8 MB cho prompt + khung JSON. `_chia_batch`
    # cắt theo byte nên nó TỰ điều tiết: ảnh nhẹ thì đủ 8, ảnh nặng thì ít hơn.
    raise ValueError("BATCH_MAX_BYTES phải nằm trong [700_000, 7_500_000]")
if DUONG_GOI == "batch" and BACKOFF_BASE < 20:
    # Chỉ ràng cho đường BATCH. Lùi ngắn hơn thời gian một batch là quay lại
    # đúng lúc suất đồng thời vẫn bận. Đo: nền 2s -> 8/18 batch chết ở 4 luồng;
    # nền 30s -> 0/18 ở 2 luồng. Đường `chat` thì request chỉ ~8s nên nền 2s là
    # đúng — ràng nó theo luật của batch là bắt nó chờ vô cớ.
    raise ValueError("BACKOFF_BASE phải ≥ 20 khi DUONG_GOI == 'batch'")
if DUONG_GOI == "chat" and MAX_WORKERS > 28:
    # 28 là mức đã chạy 6–9 giờ sạch đêm 21-09. 30 luồng thì cổng trả
    # `EOF occurred in violation of protocol` hàng loạt — đứt ở tầng TLS.
    raise ValueError("MAX_WORKERS > 28 ở đường chat: 30 luồng đã đo ra đứt TLS")

WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUNNER={RUNNER} · RUN_SCOPE={RUN_SCOPE} · model={MODEL_NAME} · "
      f"prompt={PROMPT_VARIANT} · batch={BATCH_SIZE} · max_bytes={BATCH_MAX_BYTES:,}")

In [ ]:
import subprocess, sys, os

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

# Repo là PRIVATE — `git clone` trần sẽ hỏi mật khẩu rồi TREO notebook, không
# phải báo lỗi. `GIT_ASKPASS` trả token qua stdin, `GIT_TERMINAL_PROMPT=0` biến
# mọi lần hỏi còn sót thành lỗi ngay thay vì treo. Cùng cách nb33 đã dùng.
if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    askpass = WORK_DIR / "git-askpass.sh"
    askpass.write_text(
        '#!/bin/sh\ncase "$1" in *Username*) printf x-access-token ;; '
        '*) printf %s "$GITHUB_TOKEN" ;; esac\n',
        encoding="utf-8",
    )
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update({"GITHUB_TOKEN": secrets.get_secret("GITHUB_TOKEN"),
                "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"})
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL,
                    str(REPO_DIR)], check=True, env=env)
sys.path.insert(0, str(REPO_DIR / "src"))

os.environ["TRAMMAY_API_KEY"] = secrets.get_secret(TRAMMAY_SECRET)
for name in ("MODAL_TOKEN_ID", "MODAL_TOKEN_SECRET"):
    os.environ[name] = secrets.get_secret(name)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modal"], check=True)

from mfe.contracts.ids import build_keyframe_id
from mfe.offline.bundle import BundleLayout
from mfe.offline.caption_prompt import PROMPT_VARIANTS, tach_caption_ocr
from mfe.offline.transport import ModalVolumeTransport
from mfe.online.trammay import QuotaError, TrammayClient, TrammayError, available

_PROMPT_GOC, _mac_dinh = PROMPT_VARIANTS[PROMPT_VARIANT]

# Phụ lục ÉP ĐỘ DÀI — bắt buộc cho đường batch, đừng gỡ.
#
# Đo 22-09 trên 140 khung: batch 8 ảnh làm caption ngắn đi 195 -> 148 từ, và
# điều đó SỐNG SÓT qua cả ba mức `max_tokens` (768 / 1258 / 8192). Tức không
# phải bị cắt vì hết ngân sách — model **tự chia phần** khi biết phải trả 8 mục
# trong một JSON. `≥3 gốc động từ` tụt 52,1% -> 35,6%, McNemar p = 0,0001.
#
# Mọi nút khác đã chết: cổng BỎ QUA `max_tokens` (gửi 40 vẫn trả 263 token,
# `finish_reason='stop'`), còn `top_p`/`top_k`/`stop`/`thinkingConfig` thì
# đường batch không chuyển tiếp (Đạt xác nhận). Prompt là cần gạt duy nhất.
#
# Dán phụ lục này vào: độ dài về **197 từ** (so với 194 của bản một-ảnh),
# `≥3 gốc` lên 45,0%, và McNemar còn **p = 0,1325** — không còn phân biệt được.
#
# Câu "Never invent detail" là hàng rào, KHÔNG phải câu văn thừa: ép độ dài mà
# không có nó là mời model bịa, mà `caption_audit` chỉ đếm *có nhắc* chứ không
# đếm *nhắc có đúng*. Soi tay hai khung bị kéo dài nhiều nhất (112->209 và
# 102->192 từ) thấy chi tiết thêm vào là thật: góc máy, vệt nhoè chuyển động,
# màu sọc phản quang. Nhưng mẫu 2 khung là bằng chứng để đi tiếp, không phải
# để ngừng kiểm.
# Phụ lục CHỈ dán khi đi đường batch. Ở đường `chat` (một ảnh một request) không
# có hiện tượng model tự chia phần, nên phụ lục là thứ thừa — và `chat + len180`
# là tổ hợp CHƯA ĐO. Bản trần cho đúng 194 từ, khớp 499 video sinh đêm 21-09.
PROMPT_SUFFIX = "len180" if DUONG_GOI == "batch" else ""
PHU_LUC_DO_DAI = (
    "\n\nLENGTH REQUIREMENT: write AT LEAST 180 words for this caption.\n"
    "Describe this frame with the same completeness as if it were the only "
    "image you had been given. Do not shorten, summarise, or ration detail "
    "because other images are being processed in the same request.\n"
    "Never invent detail to reach the length — if the frame is simple, spend "
    "the words on precise colours, counts, positions and on-screen text."
)
PROMPT = _PROMPT_GOC + PHU_LUC_DO_DAI if PROMPT_SUFFIX else _PROMPT_GOC

ok, vi_sao = available()
if not ok:
    raise RuntimeError(vi_sao)

# Chữ ký là `(volume_name, *, create_if_missing, version, dotenv_path)` — tên
# volume đi VỊ TRÍ, và KHÔNG có tham số `local_root`. Gọi `volume=...` ném
# TypeError ngay ở ô này.
RESEARCH = ModalVolumeTransport(RESEARCH_VOLUME, create_if_missing=False)
BTC = ModalVolumeTransport(BTC_VOLUME, create_if_missing=False)
BTC_LAYOUT = BundleLayout(BTC_BUNDLE)
CACHE_ANH.mkdir(parents=True, exist_ok=True)

# `backoff_base=30`, KHÔNG phải mặc định 2,0.
#
# Trần đồng thời của cổng đếm request ĐANG BAY, và một batch 8 ảnh chạy 23–47s.
# Lùi 2s/4s/8s là thử lại vào đúng lúc các suất vẫn chưa trống — cả bốn lần đều
# trượt. Đo 22-09 trên cùng 140 khung, 2 luồng: nền 2s -> 1/18 batch chết; nền
# 30s -> **0/18**, 140/140 caption.
client = TrammayClient.from_env(
    timeout=REQUEST_TIMEOUT, max_retries=MAX_RETRIES, backoff_base=BACKOFF_BASE
)
client = client.__class__(**{**client.__dict__, "model": MODEL_NAME})
print(f"prompt {PROMPT_VARIANT}: {len(PROMPT)} ký tự · batch={BATCH_SIZE} · client sẵn sàng")

## Chia việc — theo VIDEO, không theo keyframe

Chia theo video để mỗi worker tự đủ: một video là một đơn vị marker, và hai worker không bao giờ ghi cùng một file. Phép chia là **xác định** (băm tên video), nên chạy lại ở máy khác vẫn ra đúng cùng phần việc — không cần bảng phân công lưu ở đâu cả.

**Danh sách video đọc từ chính volume BTC**, không từ dataset Kaggle. Phép băm chia việc chạy trên danh sách đó, nên sáu worker **bắt buộc** thấy cùng một tập — mà đó chính là điều kiện để không video nào bị bỏ sót.


In [ ]:
import csv, hashlib, json

def _liet_ke_video() -> list:
    """873 video, đọc từ `map-keyframes/` trên volume BTC."""
    ra = []
    for name in BTC.listdir(f"{BTC_LAYOUT.root}/map-keyframes"):
        ten = name.rsplit("/", 1)[-1]
        if ten.endswith(".csv"):
            ra.append(ten[:-4])
    return sorted(ra)

VIDEOS = _liet_ke_video()
print(f"volume BTC: {len(VIDEOS)} video")
if len(VIDEOS) != EXPECTED_VIDEOS:
    raise RuntimeError(
        f"mong {EXPECTED_VIDEOS} video, volume trả {len(VIDEOS)} — DỪNG. "
        "Chia việc tính trên danh sách này; nó sai là có video không ai làm."
    )

def shard_cua(video_id: str) -> str:
    """Băm xác định -> tên worker. Đổi hàm này là đổi toàn bộ phân công."""
    h = hashlib.sha256(video_id.encode()).hexdigest()
    return RUNNERS[int(h[:8], 16) % len(RUNNERS)]

PHAN_CONG = {r: [v for v in VIDEOS if shard_cua(v) == r] for r in RUNNERS}
for r in RUNNERS:
    print(f"  {r}: {len(PHAN_CONG[r]):>3} video")
CUA_TOI = PHAN_CONG[RUNNER] if RUN_SCOPE != "PREFLIGHT" else PHAN_CONG[RUNNER][:1]
print(f"\n{RUNNER} xử lý {len(CUA_TOI)} video trong lượt này")

## Đọc khung — `frame_idx` lấy từ CSV, ảnh lấy theo `n`

Hai số, hai vai, và chúng **không** thay nhau được:

- **`n`** đặt tên file ảnh (`001.jpg`) — dùng để MỞ ảnh;

- **`frame_idx`** là khung thật trong video — dùng để ĐẶT KHOÁ.

`build_keyframe_id(video_id, frame_idx)` là đường duy nhất dựng khoá. Đừng tự f-string.

**614 cặp keyframe của batch 1 dùng chung một `frame_idx`** (artefact làm tròn của BTC, 0,346%). Gộp phải **giữ bản đầu tiên** để ra đúng 176.707 dòng — nếu không hai arm đánh địa chỉ khác nhau và hợp nhất cộng điểm của hai khung khác nhau.


In [ ]:
from concurrent.futures import ThreadPoolExecutor

def khung_cua(video_id: str) -> list:
    """`[(keyframe_id, n)]`, đã gộp giữ bản ĐẦU TIÊN khi trùng `frame_idx`.

    `map-keyframes` kéo từ volume BTC. Trả `n` chứ chưa trả đường dẫn ảnh: ảnh
    tải sau, song song — tải tuần tự 300 ảnh qua Modal mất hàng phút.
    """
    local_csv = WORK_DIR / "maps" / f"{video_id}.csv"
    local_csv.parent.mkdir(parents=True, exist_ok=True)
    if not local_csv.is_file():
        BTC.get(BTC_LAYOUT.map_keyframes(video_id), local_csv)
    ra, da_thay = [], set()
    with local_csv.open(encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            frame_idx = int(row["frame_idx"])
            if frame_idx in da_thay:      # 614 cặp trùng — giữ bản đầu
                continue
            da_thay.add(frame_idx)
            ra.append((build_keyframe_id(video_id, frame_idx), int(row["n"])))
    return ra

def tai_anh(video_id: str, khung: list, luong: int = 8) -> dict:
    """`{keyframe_id: Path}` — tải ảnh còn thiếu từ volume BTC, bỏ qua ảnh đã có."""
    thu_muc = CACHE_ANH / video_id
    thu_muc.mkdir(parents=True, exist_ok=True)

    def mot(item):
        kid, n = item
        dich = thu_muc / f"{n:03d}.jpg"
        if dich.is_file() and dich.stat().st_size > 0:
            return kid, dich
        try:
            BTC.get(f"{BTC_LAYOUT.keyframes_dir(video_id)}/{n:03d}.jpg", dich)
        except Exception:      # ảnh thiếu không được làm đổ cả lượt
            dich.unlink(missing_ok=True)
            return kid, None
        return kid, (dich if dich.is_file() and dich.stat().st_size else None)

    with ThreadPoolExecutor(max_workers=luong) as ex:
        return {k: p for k, p in ex.map(mot, khung) if p is not None}

_thu = khung_cua(CUA_TOI[0])
print(f"{CUA_TOI[0]}: {len(_thu)} khung · khoá mẫu {_thu[0][0] if _thu else '(rỗng)'}")
assert not _thu or "#" in _thu[0][0], "khoá phải dạng {video}#{frame_idx:06d}"

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with Path(p).open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_json(obj) -> str:
    return hashlib.sha256(
        json.dumps(obj, sort_keys=True, ensure_ascii=False).encode()
    ).hexdigest()

def duong_remote(video_id: str) -> dict:
    return {
        "captions": f"{OUTPUT_ROOT}/parts/captions/{video_id}.json",
        "marker": f"{OUTPUT_ROOT}/markers/{RUNNER}/{video_id}.json",
        # Checkpoint để ở THƯ MỤC KHÁC, không phải `parts/captions/`.
        #
        # Phiên Kaggle bị reset là mất sạch video đang làm dở (đêm 21-09: cả 6
        # worker mất video đang chạy), nên đẩy dở chừng lên volume là đáng. Nhưng
        # nếu đẩy vào ĐÚNG đường của bản hoàn chỉnh thì "có file trong
        # parts/captions/" thôi không còn nghĩa là "video đã xong" — và
        # AGGREGATE đọc đúng chỗ đó. Tách đường ra làm bất biến thành CẤU TRÚC
        # thay vì thành một phép kiểm ai đó phải nhớ viết.
        "checkpoint": f"{OUTPUT_ROOT}/checkpoints/{RUNNER}/{video_id}.json",
    }

def marker_hop_le(video_id: str):
    """Marker chỉ tính là xong khi hash của FILE khớp hash ghi trong marker.

    Marker là điểm chốt, không phải file: một part đẩy dở vẫn tồn tại trên
    volume, nên "có file" không thể có nghĩa là "đã xong".
    """
    paths = duong_remote(video_id)
    if not RESEARCH.exists(paths["marker"]):
        return None
    local = WORK_DIR / "marker-check" / f"{video_id}.json"
    local.parent.mkdir(parents=True, exist_ok=True)
    try:
        RESEARCH.get(paths["marker"], local)
        marker = json.loads(local.read_text(encoding="utf-8"))
    except Exception:
        return None
    if marker.get("status") != "COMPLETE" or marker.get("runner") != RUNNER:
        return None
    if marker.get("prompt_variant") != PROMPT_VARIANT or marker.get("model") != MODEL_NAME:
        return None   # đổi prompt/model là phải sinh lại, không dùng lại bản cũ
    # ⚠ CỐ Ý KHÔNG kiểm `prompt_suffix`.
    #
    # 499 video sinh đêm 21-09 chạy một-ảnh-một-request với `v5_bao` TRẦN, nên
    # marker của chúng không có trường này. Kiểm nó là loại sạch 99.755 caption
    # đã trả tiền và bắt sinh lại 176.707 khung thay vì 76.952.
    #
    # Chấp nhận được vì hai nửa đã được ĐO là tương đương: batch + phụ lục ra
    # 197 từ so với 194 của bản một-ảnh, `≥3 gốc` 45,0% so 52,1%, McNemar
    # p = 0,1325 — không phân biệt được ở n=140. Trường này ghi để sau này TRUY
    # được nửa nào sinh bằng gì, không phải để chặn.
    if RESEARCH.sha256(paths["captions"]) != marker.get("captions_sha256"):
        return None
    return marker

def mot_lo(lo):
    """Một ĐƠN VỊ công việc — `[(kid, đường_ảnh)]` — trả `[(kid, caption|None)]`.

    Hai đường, cùng một hợp đồng ra/vào, nên vòng ngoài không cần biết đang đi
    đường nào. Đó là điểm gộp: mọi thứ khác — đếm hỏng, ghi checkpoint, marker,
    PARTIAL — dùng chung một nhánh mã, không tách đôi.

    Đường `"chat"`: mỗi phần tử là một request riêng, hỏng thì mất ĐÚNG một ảnh.
    Đường `"batch"`: cả lô là một request, hỏng thì mất cả lô (xem `mot_batch`).
    """
    if DUONG_GOI == "chat":
        kid, anh = lo[0]
        try:
            return [(kid, client.chat(PROMPT, images=[anh],
                                      max_tokens=MAX_TOKENS_CHAT, temperature=0.0))]
        except QuotaError:
            raise                     # hết hạn mức: ném lên, ĐỪNG nuốt
        except TrammayError as e:
            print(f"  ảnh hỏng {kid}: {str(e)[:120]}")
            return [(kid, None)]
    return mot_batch(lo)

def mot_batch(batch):
    """Một request cho nhiều ảnh. Khoá đi theo `id`, KHÔNG theo thứ tự.

    `chat_batch` đối chiếu tập id trả về với tập id gửi đi và ném nếu lệch,
    nên caption không thể gắn nhầm sang khung khác. Nó cũng ném khi có item
    chạm trần `max_tokens` (`finish_reason == "length"`) — caption cụt vẫn là
    chuỗi hợp lệ, không phép kiểm nội dung nào nhìn ra.

    Hỏng thì trả None cho CẢ batch: nhận phần lành của một batch đã lỗi là ghi
    vào kho thứ mình không kiểm được.
    """
    try:
        ra = client.chat_batch(
            [{"id": kid, "prompt": PROMPT, "images": [anh]} for kid, anh in batch],
            max_tokens=MAX_TOKENS * len(batch),
            temperature=0.0,
        )
        return [(kid, ra.get(kid)) for kid, _ in batch]
    except QuotaError:
        raise                         # hết hạn mức: ném lên, ĐỪNG nuốt
    except (TrammayError, ValueError) as e:
        # Client đã tự thử lại MAX_RETRIES lần; tới đây là hết đường cho batch
        # này. Vòng ngoài ĐẾM nó thay vì để cả video chết.
        print(f"  batch {len(batch)} ảnh hỏng: {str(e)[:160]}")
        return [(kid, None) for kid, _ in batch]

def _uoc_byte(item) -> int:
    """Ước lượng thân JSON sau base64 — base64 phình 4/3 so với file gốc."""
    _kid, anh = item
    return ((Path(anh).stat().st_size + 2) // 3) * 4 + len(PROMPT.encode()) + 1024

def _chia_batch(items, size: int, max_bytes: int):
    """Cắt theo CẢ số lượng lẫn byte. Byte mới là hàng rào thật.

    Trần thân request của cổng là 2 MB (đo 22-09: 1,9 MB qua, 2,1 MB trả 413).
    Ảnh keyframe p90 là 314 KB → base64 419 KB, nên bốn ảnh xấu nhất đã 1,7 MB.
    Cắt theo số lượng thôi là thỉnh thoảng ăn 413 — mà 413 KHÔNG nằm trong
    `RETRY_CODES` nên nó ném thẳng, mất cả batch.
    """
    ra, hien_tai, byte_hien_tai = [], [], 0
    for item in items:
        b = _uoc_byte(item)
        if hien_tai and (len(hien_tai) >= size or byte_hien_tai + b > max_bytes):
            ra.append(hien_tai)
            hien_tai, byte_hien_tai = [], 0
        hien_tai.append(item)
        byte_hien_tai += b
    if hien_tai:
        ra.append(hien_tai)
    return ra

def xu_ly_video(video_id: str, publish: bool) -> dict:
    khung = khung_cua(video_id)
    local = WORK_DIR / "parts" / video_id
    local.mkdir(parents=True, exist_ok=True)
    file_cap = local / "captions.json"
    remote = duong_remote(video_id)
    # Phiên Kaggle có thể bị reset giữa chừng. Kéo checkpoint về trước khi gọi
    # API, nếu không mọi khung của video đang dở sẽ bị GỬI LẠI — tốn đúng thứ
    # quota đang thiếu.
    if not file_cap.is_file() and publish and RESEARCH.exists(remote["checkpoint"]):
        try:
            RESEARCH.get(remote["checkpoint"], file_cap)
            if not isinstance(json.loads(file_cap.read_text(encoding="utf-8")), dict):
                file_cap.unlink(missing_ok=True)
        except Exception:
            file_cap.unlink(missing_ok=True)
    caps = json.loads(file_cap.read_text(encoding="utf-8")) if file_cap.is_file() else {}
    # Chỉ tải ảnh cho khung CHƯA có caption — chạy lại không tải lại phần đã xong.
    anh = tai_anh(video_id, [(k, n) for k, n in khung if k not in caps])
    con = list(anh.items())
    # Đường `chat`: mỗi ảnh một lô, nên hỏng một lượt chỉ mất một ảnh chứ
    # không mất cả chùm. Đường `batch`: cắt theo byte trước, số lượng sau.
    lo_viec = ([[x] for x in con] if DUONG_GOI == "chat"
               else _chia_batch(con, BATCH_SIZE, BATCH_MAX_BYTES))

    t0, xong, hong, so_batch = time.time(), 0, 0, 0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        for ket_qua in ex.map(mot_lo, lo_viec):
            for kid, text in ket_qua:
                if text is None:
                    hong += 1
                    continue
                # Lưu THÔ cả hai khối. Tách ở AGGREGATE, không tách ở đây:
                # tách sai thì re-split từ part tốn 0 đồng, còn sinh lại
                # cả kho tốn ~320 triệu token.
                caps[kid] = text.strip()
                xong += 1
            so_batch += 1
            # Mốc ghi tính theo ẢNH, không theo lô: 80 ảnh là 10 batch ở
            # đường batch và 80 lượt ở đường chat — cùng một mức rủi ro.
            if so_batch % MOC_GHI == 0:
                file_cap.write_text(json.dumps(caps, ensure_ascii=False), encoding="utf-8")
                if publish:
                    RESEARCH.put(file_cap, remote["checkpoint"])
    file_cap.write_text(json.dumps(caps, ensure_ascii=False), encoding="utf-8")

    bao_cao = {"video_id": video_id, "khung": len(khung), "caption": len(caps),
               "hong": hong, "lo": len(lo_viec), "giay": round(time.time() - t0, 1)}
    if not publish:
        return bao_cao
    if len(caps) != len(khung):
        bao_cao["status"] = "PARTIAL"   # thiếu khung thì KHÔNG ghi marker
        return bao_cao

    RESEARCH.put(file_cap, remote["captions"])
    if RESEARCH.sha256(remote["captions"]) != sha256_file(file_cap):
        raise RuntimeError(f"{video_id}: captions hash mismatch sau khi đẩy")
    marker = {"schema_version": 1, "status": "COMPLETE", "runner": RUNNER,
              "video_id": video_id, "frames": len(caps), "model": MODEL_NAME,
              "prompt_variant": PROMPT_VARIANT, "prompt_suffix": PROMPT_SUFFIX,
              "duong_goi": DUONG_GOI,
              "max_tokens": MAX_TOKENS, "batch_size": BATCH_SIZE,
              "captions_sha256": sha256_file(file_cap),
              "heldout_accessed": False, "btc_mutated": False,
              "completed_at_utc": datetime.now(timezone.utc).isoformat()}
    marker["marker_payload_hash"] = sha256_json(marker)
    mp = local / "marker.json"
    mp.write_text(json.dumps(marker, ensure_ascii=False), encoding="utf-8")
    RESEARCH.put(mp, remote["marker"])      # marker ghi SAU CÙNG
    if RESEARCH.sha256(remote["marker"]) != sha256_file(mp):
        raise RuntimeError(f"{video_id}: marker hash mismatch")
    bao_cao["status"] = "COMPLETE"
    return bao_cao

In [ ]:
if RUN_SCOPE in {"PREFLIGHT", "SHARD"}:
    xong_roi = ([v for v in CUA_TOI if marker_hop_le(v)]
                if RUN_SCOPE == "SHARD" else [])
    con_lai = [v for v in CUA_TOI if v not in set(xong_roi)]
    print(f"{len(xong_roi)} video đã COMPLETE · còn {len(con_lai)}")

    bao_caos, that_bai = [], []
    for i, video_id in enumerate(con_lai, 1):
        try:
            r = xu_ly_video(video_id, publish=RUN_SCOPE == "SHARD")
            bao_caos.append(r)
            print(f"[{i}/{len(con_lai)}] {video_id}: {r['caption']}/{r['khung']} "
                  f"· hỏng {r['hong']} · {r['giay']}s · {r.get('status', 'PREFLIGHT')}")
        except QuotaError as e:
            # DỪNG HẲN, không đi tiếp. Quét nốt danh sách khi hạn mức đã cạn
            # chỉ tạo ra một loạt video PARTIAL và đốt hết phiên Kaggle — đo
            # 23-09: 1.419,9s cho MỘT video mà thu về 0 caption.
            print(f"\n⛔ HẾT HẠN MỨC ở video {i}/{len(con_lai)} ({video_id})")
            print(f"   {str(e)[:200]}")
            print("   DỪNG lượt chạy. Phần đã xong giữ nguyên marker; chạy lại")
            print("   sau khi cửa sổ hạn mức reset (nhà cung cấp: 5 tiếng).")
            break
        except Exception as e:
            that_bai.append({"video_id": video_id, "loi": repr(e)})
            print(f"[{i}/{len(con_lai)}] {video_id}: THẤT BẠI {e!r}")

    tong_hong = sum(r["hong"] for r in bao_caos)
    tong_cap = sum(r["caption"] for r in bao_caos)
    print(f"\nTổng: {tong_cap:,} caption · {tong_hong} khung hỏng "
          f"({tong_hong / max(1, tong_cap + tong_hong) * 100:.1f}%)")
    if tong_hong / max(1, tong_cap + tong_hong) > 0.02:
        print("⚠ >2% hỏng — xem dòng `hỏng` ở trên để biết vì sao:\n"
              "   429 quota -> DỪNG, đợi cửa sổ hạn mức reset (5 tiếng)\n"
              "   HTTP 413  -> hạ BATCH_MAX_BYTES\n"
              "   max_tokens -> nới MAX_TOKENS\n"
              "   EOF/TLS    -> hạ MAX_WORKERS")

In [ ]:
if RUN_SCOPE == "AGGREGATE":
    gop, trung = {}, 0
    thieu_runner = []

    def _marker_cua(runner: str, video_id: str):
        """Marker của MỘT runner cụ thể — `marker_hop_le` chỉ soi runner của mình.

        ⚠ Ô này TỪNG chỉ hỏi `RESEARCH.exists(parts/captions/...)`. Hồi đó an
        toàn vì part chỉ lên volume SAU khi video xong. Nhưng "có file" là một
        bất biến người khác phải giữ hộ, và nó đã suýt vỡ khi có người thêm
        đường đẩy checkpoint vào đúng thư mục ấy (§5: "marker là điểm chốt,
        không phải file"). Giờ hỏi thẳng marker, và checkpoint nằm ở thư mục
        riêng — hai lớp, mỗi lớp đủ một mình.
        """
        p = f"{OUTPUT_ROOT}/markers/{runner}/{video_id}.json"
        if not RESEARCH.exists(p):
            return None
        lp = WORK_DIR / "gop-marker" / f"{video_id}.json"
        lp.parent.mkdir(parents=True, exist_ok=True)
        try:
            RESEARCH.get(p, lp)
            m = json.loads(lp.read_text(encoding="utf-8"))
        except Exception:
            return None
        if m.get("status") != "COMPLETE" or m.get("video_id") != video_id:
            return None
        if m.get("prompt_variant") != PROMPT_VARIANT or m.get("model") != MODEL_NAME:
            return None
        return m

    for r in RUNNERS:
        for video_id in PHAN_CONG[r]:
            remote = f"{OUTPUT_ROOT}/parts/captions/{video_id}.json"
            m = _marker_cua(r, video_id)
            if m is None or not RESEARCH.exists(remote):
                thieu_runner.append((r, video_id))
                continue
            # Hash trong marker là thứ phân biệt "file đã đẩy xong" với "file
            # đang đẩy dở". Lệch hash = coi như chưa có, KHÔNG gộp.
            if RESEARCH.sha256(remote) != m.get("captions_sha256"):
                print(f"⚠ {video_id}: hash lệch marker — bỏ qua, chạy lại SHARD của {r}")
                thieu_runner.append((r, video_id))
                continue
            local = WORK_DIR / "gop" / f"{video_id}.json"
            local.parent.mkdir(parents=True, exist_ok=True)
            RESEARCH.get(remote, local)
            for k, v in json.loads(local.read_text(encoding="utf-8")).items():
                if k in gop:
                    trung += 1     # hai worker cùng ghi một khoá = phân công hỏng
                    continue
                gop[k] = v

    print(f"gộp được {len(gop):,} caption · trùng khoá {trung} · thiếu {len(thieu_runner)} video")
    if trung:
        raise RuntimeError(f"{trung} khoá trùng — hai worker chồng lấn, KHÔNG được gộp")
    if len(gop) != EXPECTED_FRAMES:
        print(f"⚠ mong {EXPECTED_FRAMES:,}, được {len(gop):,} "
              f"(lệch {len(gop) - EXPECTED_FRAMES:+,}) — kiểm trước khi dùng")

    # Tên theo MODEL SINH RA NÓ, không theo tên file kho đang đọc. Đặt
    # `qwen_captions.json` thì tiện thay kho hơn một bước, nhưng nội dung là
    # Gemini — và hai tháng sau không ai phân biệt được bản nào là bản nào.
    # Tách MỘT lần, ở đây, bằng đúng hàm mà người sinh prompt đã viết.
    caps_only, ocr_only, thieu_ocr = {}, {}, 0
    for k, v in gop.items():
        c, o = tach_caption_ocr(v)
        caps_only[k] = c
        if o:
            ocr_only[k] = o
        else:
            thieu_ocr += 1
    print(f"tách: {len(caps_only):,} caption · {len(ocr_only):,} có OCR · "
          f"{thieu_ocr:,} khung không có chữ trên màn hình "
          f"({thieu_ocr / max(len(gop), 1) * 100:.1f}%)")
    ra = WORK_DIR / "gemini_captions.json"
    ra.write_text(json.dumps(caps_only, ensure_ascii=False), encoding="utf-8")
    RESEARCH.put(ra, f"{OUTPUT_ROOT}/gemini_captions.json")
    ra_ocr = WORK_DIR / "gemini_ocr.json"
    ra_ocr.write_text(json.dumps(ocr_only, ensure_ascii=False), encoding="utf-8")
    RESEARCH.put(ra_ocr, f"{OUTPUT_ROOT}/gemini_ocr.json")
    # Bản THÔ ở lại volume: tách lại không cần gọi API lần nào.
    ra_tho = WORK_DIR / "gemini_raw.json"
    ra_tho.write_text(json.dumps(gop, ensure_ascii=False), encoding="utf-8")
    RESEARCH.put(ra_tho, f"{OUTPUT_ROOT}/gemini_raw.json")
    manifest = {"schema_version": 1,
                "status": "COMPLETE" if len(gop) == EXPECTED_FRAMES else "PARTIAL",
                "frames": len(gop), "frames_with_ocr": len(ocr_only),
                "model": MODEL_NAME,
                "prompt_variant": PROMPT_VARIANT, "prompt_suffix": PROMPT_SUFFIX,
              "duong_goi": DUONG_GOI,
                "max_tokens": MAX_TOKENS,
                "missing_videos": [v for _, v in thieu_runner],
                "captions_sha256": sha256_file(ra),
                "heldout_accessed": False, "btc_mutated": False,
                "created_at_utc": datetime.now(timezone.utc).isoformat()}
    manifest["manifest_payload_hash"] = sha256_json(manifest)
    mp = WORK_DIR / "manifest.json"
    mp.write_text(json.dumps(manifest, ensure_ascii=False), encoding="utf-8")
    RESEARCH.put(mp, f"{OUTPUT_ROOT}/manifest.json")
    print(f"AGGREGATE {manifest['status']}: {len(gop):,} caption -> {OUTPUT_ROOT}")

## Sau khi AGGREGATE

1. **Đếm lại**: `gemini_captions.json` phải có đúng **176.707** khoá. Lệch là có worker chồng lấn hoặc sót video — ô AGGREGATE đã ném lỗi khi thấy khoá trùng.

   ⚠ **Thay vào kho là một bước THỦ CÔNG, có chủ ý.** Tải về rồi chép đè lên `stores/flat/data/vectors/qwen_captions.json` — arm vẫn tên `caption_qwen` vì lý do lịch sử, và đổi tên arm là việc riêng (nó chạm bảng trọng số, `TEXT_ARM_SPECS` và test). **Giữ lại bản Qwen cũ** trước khi chép đè: nó là phía đối chứng duy nhất, mất nó là không dựng lại được con số nền.

2. **Chấm chất lượng** trước khi thay kho, bằng chính thước của repo:

```python
from mfe.offline.caption_audit import audit, format_audit
print(format_audit(audit((c, "") for c in gop.values()), "Gemini v5_bao"))
```
Hàng cần nhìn là **`≥3 gốc động từ`**, không phải hàng `quan hệ` có/không — hàng đó bão hoà sớm và không phân biệt được gì. Mốc để so: Qwen+aic **5,0%**, Gemini+v5_bao trên mẫu 200 ảnh **49,7%**.

3. **`bge_m3` phải nhúng lại.** `dense_vectors.npz` là nhúng của caption CŨ; giữ nguyên nó cạnh caption mới là hai arm mô tả hai thứ khác nhau dưới cùng một khoá.

4. **Leo dốc lại bảng trọng số, và XOÁ `--evidence-cache` cũ.** Vân tay của cache chỉ bắt config/queryset/topk/en-source — nó **không** bắt được việc đổi nội dung kho. Dùng lại cache cũ là leo dốc trên kết quả truy hồi của kho cũ mà vẫn in ra một bảng đẹp.

5. **`caption_qwen` giờ phải ăn truy vấn ĐÃ DỊCH** (`query_language="en"`, sửa 21-09). Đo trên 163 câu dev: truy vấn tiếng Việt trúng 27/163, bản dịch trúng **122/163**.
